In [0]:
source = spark.conf.get("source")

In [0]:
import dlt

# Bronze Layer
@dlt.table(
name="bronze_user"
)
def bronze_table():
  return (
    spark.readStream.format("cloudFiles") \
      .option("cloudFiles.format", "parquet") \
      .option("cloudFiles.inferColumnTypes", True) \
      .load(f"{source}")
  )


 

In [0]:
def process_rescue_data(df, target_schema: StructType):
    df = df.withColumn("_rescued_data_modified", from_json(col("_rescued_data"), MapType(StringType(), StringType())))
    for field in target_schema.fields:
        data_type = field.dataType
        column_name = field.name
        # Check if "_rescue_data" is not null and if the key exists
        key_condition = expr(f"_rescued_data_modified IS NOT NULL AND map_contains_key(_rescued_data_modified, '{column_name}')")
        # Extract the rescued value for this column, if it exists, and cast it to the target data type
        rescued_value = when(key_condition,                
col("_rescued_data_modified").getItem(column_name).cast(data_type)).otherwise(col(column_name).cast(data_type))
        # Update the DataFrame with the merged column
        df = df.withColumn(column_name, rescued_value)
        df = df.withColumn(column_name, col(column_name).cast(data_type))
    df = df.drop('_rescued_data_modified')

    # Setting the _rescued_data to null after processing since we use the column to check qualit expectation for schema update
    df = df.withColumn('_rescued_data', lit(None).cast(StringType()))
    return df​

In [0]:
from pyspark.sql.types import StructType, StructField, DoubleType

updated_datatypes = StructType([
  StructField("discount", DoubleType(), True)
])
@dlt.table(
    name=f"silver_user"
)
@dlt.expect("schema_update_rule", "_rescued_data is null")
def silver_table():
    return (
    process_rescue_data(dlt.read_stream('bronze_user'), updated_datatypes)
    )